# Query Web IQ through MCP

Connect directly to the Web IQ MCP endpoint, inspect the tools it exposes, and call them to search the web, images, and videos. Web IQ authenticates with an API key that you send in the `x-apikey` header.

## Step 1: Load configuration and create an MCP session helper

Load the Web IQ key from the project `.env` file. If `WEB_IQ_KEY` is missing, get the key for your Web IQ account from the Web IQ portal and add it to `.env`.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **ignite26-prel42** environment with the path **.venv\Scripts\python.exe**.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import os
from contextlib import asynccontextmanager

import httpx
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

load_dotenv(override=True)

WEB_IQ_KEY = os.environ["WEB_IQ_KEY"]
WEB_IQ_MCP_URL = "https://api.microsoft.ai/v3/mcp"


@asynccontextmanager
async def open_webiq_session():
    async with httpx.AsyncClient(
        headers={"x-apikey": WEB_IQ_KEY},
        timeout=httpx.Timeout(30, read=120),
    ) as http_client:
        async with streamable_http_client(WEB_IQ_MCP_URL, http_client=http_client) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session

print("Web IQ configuration loaded")

## Step 2: List the available tools

Open an MCP session and inspect every tool that Web IQ exposes. Each tool covers a different kind of web content, such as web pages, news, images, and videos. Your Web IQ profile controls which of them you can call: the lab profile allows web, image, and video search.

In [ ]:
from rich.console import Console

console = Console()

async with open_webiq_session() as session:
    tools_response = await session.list_tools()

for tool in tools_response.tools:
    console.print(f"[bold cyan]{tool.name}[/]: {tool.description}")
    console.print(f"  Parameters: {', '.join(tool.inputSchema.get('properties', {}))}")

## Step 3: Search the web

Call the `web` tool with a query. Each result has a title, URL, and content. The notebook sets `contentFormat` to `passage`, which returns the part of each page that is most relevant to the query. Passages are the recommended format for grounding, and other formats require full-page content access on your Web IQ profile. Change `query` to try other searches.

In [ ]:
query = "mental health benefits trends for employers"

async with open_webiq_session() as session:
    result = await session.call_tool("web", {"query": query, "maxResults": 5, "contentFormat": "passage"})

for page in result.structuredContent["webResults"]:
    console.print(f"[bold]{page['title']}[/]")
    console.print(f"[link={page['url']}]{page['url']}[/link]")
    console.print(page["content"][:400] + "...\n")

## Step 4: Search for images

The `images` tool returns images with a caption and the page that hosts each image. Agents can use the captions to pick an image that matches a request.

In [ ]:
query = "how to use a claw hammer"

async with open_webiq_session() as session:
    result = await session.call_tool("images", {"query": query, "maxResults": 5})

for image in result.structuredContent["imageResults"]:
    console.print(f"[bold]{image['title']}[/]")
    console.print(image.get("caption", ""))
    console.print(f"[link={image['hostPageUrl']}]{image['hostPageUrl']}[/link]\n")

## Step 5: Search for videos

The `videos` tool returns videos with their publisher, length, and view count.

In [ ]:
query = "how to use a claw hammer"

async with open_webiq_session() as session:
    result = await session.call_tool("videos", {"query": query, "maxResults": 5})

for video in result.structuredContent["videoResults"]:
    console.print(f"[bold]{video['title']}[/]")
    console.print(f"{video.get('publishedBy', '')} | {video.get('length', '')} | {video.get('viewCount', 0):,} views")
    console.print(f"[link={video['url']}]{video['url']}[/link]\n")

## ✅ Mission complete

**What you built:**

* ✓ **Authenticated MCP session**: A direct connection to the Web IQ MCP endpoint with your API key.
* ✓ **Web, image, and video search**: Current results from each Web IQ search tool.

In the Foundry IQ exercise, `foundryiq-webiq.ipynb` adds the same MCP endpoint as a knowledge source, so a knowledge base can combine web results with your own documents.

➡️ Return to lab instructions.